In [1]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
import zipfile

# --- GIẢI NÉN FILE Hantu.zip ---
zip_path = "Hantu.zip"
extract_dir = "html_input"
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"✅ Đã giải nén {zip_path} vào thư mục {extract_dir}")
else:
    print(f"❌ Không tìm thấy file {zip_path}")

✅ Đã giải nén Hantu.zip vào thư mục html_input


In [ ]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
import unicodedata
from tqdm import tqdm

# --- 1. HÀM LÀM SẠCH TIẾNG PALI ---
def clean_pali_display(text):
    """Xóa chú thích rác, số trang PTS để hiển thị"""
    if not text: return ""
    # Xóa số trang PTS thường có dạng [M.i.1] hoặc [1]
    text = re.sub(r'\[.*?\]', '', text)
    # Xóa các con số đứng một mình ở đầu dòng
    text = re.sub(r'^[0-9\.]+\s', '', text)
    return text.strip()

def clean_pali_strictly(text):
    """Làm sạch sâu để AI so sánh (giữ diacritics ā, ī, ū, ṃ...)"""
    if not text: return ""
    # Chuẩn hóa Unicode NFC để tránh lỗi tách dấu
    text = unicodedata.normalize('NFC', text).lower()
    # Chỉ giữ lại chữ cái Pali và khoảng trắng, xóa mọi dấu câu
    text = re.sub(r'[,\.\?\!\:\;"\(\)\[\]\{\}\-–—…\t\n\r\']+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# --- 2. QUY TRÌNH XỬ LÝ CHÍNH ---
pali_data = []
input_root = "pali_input"

# Lấy danh sách file và sắp xếp
all_files = []
for root, dirs, f_names in os.walk(input_root):
    for f in f_names:
        if f.lower().endswith(('.html', '.htm')):
            all_files.append(os.path.join(root, f))
all_files.sort()

print(f"🚀 Đang bóc tách {len(all_files)} file Pali...")

for path in tqdm(all_files):
    # Lấy số kinh từ tên file (ví dụ mn001.html -> 1)
    file_name = os.path.basename(path)
    sutta_no_match = re.findall(r'\d+', file_name)
    sutta_no = int(sutta_no_match[0]) if sutta_no_match else 0

    if sutta_no == 0: continue # Bỏ qua nếu không xác định được số kinh

    with open(path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    # 1. Tìm tên kinh (thường nằm trong thẻ h1 hoặc title)
    sutta_name = ""
    h1 = soup.find('h1')
    if h1: sutta_name = h1.get_text().strip()
    else: sutta_name = f"MN {sutta_no}"

    # 2. Lấy nội dung theo từng khối văn bản (p, div, span)
    # Trong các file Pali chuẩn, nội dung thường nằm trong thẻ <p>
    paragraphs = soup.find_all(['p', 'div'])
    para_idx = 1

    for p in paragraphs:
        raw_text = p.get_text().strip()

        # Lọc bỏ các dòng quá ngắn (thường là rác hoặc tiêu đề phụ)
        if len(raw_text) < 15: continue

        # Bỏ qua các dòng chứa thông tin bản quyền (thường ở cuối file)
        if any(x in raw_text.lower() for x in ["copyright", "license", "vri", "suttacentral"]):
            continue

        content_disp = clean_pali_display(raw_text)
        content_clean = clean_pali_strictly(content_disp)

        if len(content_clean.split()) > 4: # Chỉ lấy các câu có trên 4 từ
            pali_data.append({
                "Sutta_No": sutta_no,
                "Sutta_Name": sutta_name,
                "Segment_ID": f"MN.{sutta_no}.{para_idx}",
                "Content": content_disp,
                "Cleaned_Content": content_clean
            })
            para_idx += 1

# --- 3. XUẤT CSV ---
if not pali_data:
    print("❌ Lỗi: Không lấy được dữ liệu Pali. Kiểm tra lại cấu trúc HTML.")
else:
    df_pali = pd.DataFrame(pali_data)
    # Loại bỏ trùng lặp nếu có
    df_pali = df_pali.drop_duplicates(subset=['Segment_ID'])

    output_name = "Pali_MN_Full_152.csv"
    df_pali.to_csv(output_name, index=False, encoding='utf-8-sig')

    print("\n" + "="*40)
    print(f"📊 THỐNG KÊ PALI:")
    print(f"✅ Tổng số bài kinh bóc tách: {df_pali['Sutta_No'].nunique()}")
    print(f"📝 Tổng số phân đoạn (segments): {len(df_pali)}")
    print("="*40)
    print(f"📁 File đã lưu: {output_name}")

🚀 Đang bóc tách 60 file Pali...


  0%|          | 0/60 [00:00<?, ?it/s]

100%|██████████| 60/60 [00:01<00:00, 43.36it/s]


📊 THỐNG KÊ PALI:
✅ Tổng số bài kinh bóc tách: 1
📝 Tổng số phân đoạn (segments): 29
📁 File đã lưu: Pali_MN_Full_152.csv
